In [ ]:
!pip -q install pandas tqdm torch transformers emoji sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 6.6 MB/s eta 0:00:00


In [ ]:
import os, re, emoji, torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DATA_PATH = "gabungan_tweet_komentar_text_bersih.csv"
TEXT_COL = "text"
MODEL_NAME = "Aardiiiiy/indobertweet-base-Indonesian-sentiment-analysis"
BATCH_SIZE = 32

In [ ]:
# Upload CSV jika file belum ada di Colab
if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded))

df = pd.read_csv(DATA_PATH)
df = df[[TEXT_COL]].rename(columns={TEXT_COL: "text"})
df = df.dropna(subset=["text"]).drop_duplicates("text").reset_index(drop=True)

print("Jumlah data:", len(df))
df.head()

Jumlah data: 6902


,text
0,Media Asing Sorot Prabowo Ganti Kepala BGN Seb...
1,@txtdrimedia Berikut ini perkiraan penyebab ke...
2,@TeddGus Santai aja @TeddGus .. ngapain lw res...
3,@Ogaman888 Kalo orang pinter dan ga rakus ide ...
4,@mfsa118 kata pemilihnya karna beliau backgrou...


In [ ]:
def preprocessing_awal(text):
    text = str(text)
    text = emoji.demojize(text, language="id")
    text = re.sub(r":([a-zA-Z0-9_+\-]+):", lambda m: " " + m.group(1).replace("_", " ") + " ", text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " HTTPURL ", text)
    text = re.sub(r"@\w+", " @USER ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_preprocessing_awal"] = df["text"].apply(preprocessing_awal)
df.to_csv("01_preprocessing_awal_indobertweet.csv", index=False)

df[["text", "text_preprocessing_awal"]].head()

,text,text_preprocessing_awal
0,Media Asing Sorot Prabowo Ganti Kepala BGN Seb...,media asing sorot prabowo ganti kepala bgn seb...
1,@txtdrimedia Berikut ini perkiraan penyebab ke...,@USER berikut ini perkiraan penyebab kematian ...
2,@TeddGus Santai aja @TeddGus .. ngapain lw res...,@USER santai aja @USER .. ngapain lw respon ne...
3,@Ogaman888 Kalo orang pinter dan ga rakus ide ...,@USER kalo orang pinter dan ga rakus ide itu m...
4,@mfsa118 kata pemilihnya karna beliau backgrou...,@USER kata pemilihnya karna beliau background ...


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()


def map_label(label):
    label = str(label).lower()
    if "neg" in label or label in ["label_0", "0"]:
        return "negatif"
    if "neu" in label or label in ["label_1", "1"]:
        return "netral"
    if "pos" in label or label in ["label_2", "2"]:
        return "positif"
    return label

id2label = {int(k): map_label(v) for k, v in model.config.id2label.items()}
id2label

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/994 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{0: 'negatif', 1: 'netral', 2: 'positif'}

In [ ]:
def predict_sentiment(texts, batch_size=32):
    labels, scores = [], []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = list(texts[i:i + batch_size])
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)

        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=-1)

        pred = probs.argmax(dim=1).cpu().tolist()
        conf = probs.max(dim=1).values.cpu().tolist()
        labels.extend([id2label[p] for p in pred])
        scores.extend(conf)

    return labels, scores


df["label_sentimen"], df["confidence"] = predict_sentiment(df["text_preprocessing_awal"], BATCH_SIZE)
df.to_csv("02_hasil_labeling_indobertweet.csv", index=False)

df["label_sentimen"].value_counts()

  0%|          | 0/216 [00:00<?, ?it/s]

,count
label_sentimen,
negatif,4402
netral,2005
positif,495


In [ ]:
df[["text", "text_preprocessing_awal", "label_sentimen", "confidence"]].head(10)

,text,text_preprocessing_awal,label_sentimen,confidence
0,Media Asing Sorot Prabowo Ganti Kepala BGN Seb...,media asing sorot prabowo ganti kepala bgn seb...,netral,0.997109
1,@txtdrimedia Berikut ini perkiraan penyebab ke...,@USER berikut ini perkiraan penyebab kematian ...,negatif,0.992115
2,@TeddGus Santai aja @TeddGus .. ngapain lw res...,@USER santai aja @USER .. ngapain lw respon ne...,negatif,0.965767
3,@Ogaman888 Kalo orang pinter dan ga rakus ide ...,@USER kalo orang pinter dan ga rakus ide itu m...,netral,0.691961
4,@mfsa118 kata pemilihnya karna beliau backgrou...,@USER kata pemilihnya karna beliau background ...,negatif,0.969715
5,@bapinski13 @chococookiesess Food-waste parah....,@USER @USER food-waste parah. anakku dapet jat...,negatif,0.997713
6,@Grock230 1. MBG bikin banyak terjadi keracuna...,@USER 1. mbg bikin banyak terjadi keracunan. 2...,negatif,0.992516
7,@Bambangmulyonoo Mbg ga ada manfaatnya bisa me...,@USER mbg ga ada manfaatnya bisa membuat murid...,negatif,0.997187
8,@MiskinTV_ Rekam jejak MBG Makanan 'Bergizi' '...,@USER rekam jejak mbg makanan 'bergizi' 'grati...,negatif,0.997743
9,@99propaganda Lo aja yg bloon klu semua kerjaa...,@USER lo aja yg bloon klu semua kerjaannya di ...,negatif,0.996004


In [ ]:
try:
    from google.colab import files
    files.download("01_preprocessing_awal_indobertweet.csv")
    files.download("02_hasil_labeling_indobertweet.csv")
except Exception:
    print("File tersimpan di folder kerja notebook.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>